# Customer Churn Prediction - Feature Engineering
### Based on: Matuszelanski & Kopczewska (2022) - Customer Churn in Retail E-Commerce Business: Spatial and Machine Learning Approach
### Journal: Journal of Theoretical and Applied Electronic Commerce Research

In [0]:
from pyspark.sql.functions import (
    col, count, when, datediff, current_date,
    max as spark_max, sum as spark_sum,
    avg, lit
)
from pyspark.sql.functions import least, greatest
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
import pandas as pd

print("imports done")

**Read Silver Tables**

In [0]:
orders_df = spark.table("bharatmart.silver.slv_orders")
sessions_df = spark.table("bharatmart.silver.slv_sessions")
cart_df = spark.table("bharatmart.silver.slv_cart_events")
reviews_df = spark.table("bharatmart.silver.slv_reviews")
refunds_df = spark.table("bharatmart.silver.slv_refunds")
payments_df = spark.table("bharatmart.silver.slv_payments")

print("tables loaded")

**Drop Ghost Orders**
- EDA says - customer_id is 2% NULL in orders, drop these ghost rows before anything else.
- Paper says - clean base table is the first step before any feature computation.

In [0]:
# customer_id NULL = ghost orders, no customer attached
clean_orders = orders_df.filter(col("customer_id").isNotNull())

before = orders_df.count()
after = clean_orders.count()

print(f"before: {before:,}")
print(f"after: {after:,}")
print(f"dropped: {before - after:,} ghost rows")

we dropped 32,284 ghost orders - rows where no customer_id was attached.
these are orders that exist in the system but cannot be linked to any customer.
they could be cancelled orders, system test orders, or data pipeline errors.

keeping these rows would corrupt every customer-level feature we compute.
if a ghost order gets counted in a customer's total_spent or total_orders,
the model gets wrong behavioral signals for that customer.

the paper by Matuszelanski & Kopczewska (2022) emphasizes that clean 
base data is the foundation of reliable churn prediction.
they specifically note that behavioral features only have predictive power 
when they accurately represent actual customer behavior - 
garbage rows in means garbage features out.

we now have 1,589,411 clean orders to build features from.

**Recompute Total Amount**
- EDA says - total_amount is 99% NULL, recompute as order_amount - discount_amount.
- Paper says - monetary value (M in RFM) is one of the top 3 churn predictors.

In [0]:
# total_amount was 99% NULL in silver — recompute it ourselves
clean_orders = clean_orders.withColumn(
    "total_amount", col("order_amount") - col("discount_amount")
)

display(clean_orders.select("order_amount", "discount_amount", "total_amount").limit(5))

total_amount is now correctly computed as order_amount minus discount_amount.
this was 99% NULL in the silver layer - only 15,000 out of 1.6M rows had it populated.

by recomputing it ourselves we now have total_amount for all 1.589M clean orders.
this is critical because total_amount feeds directly into total_spent -
the monetary value feature, the M in RFM.

the paper by Matuszelanski & Kopczewska (2022) identifies monetary value 
as one of the three core RFM features that account for 60-70% of the 
model's predictive power. losing this feature would cripple the model.

discount_amount is also preserved here - customers who consistently 
get large discounts may behave differently in terms of loyalty.
we will explore this as a feature later.

**RFM Base Features**
- EDA says - recency_days, total_orders are the R and F of RFM. Clip total_orders at p99.
- Paper says - RFM features are the foundation, recency is the single strongest predictor.

In [0]:
rfm = clean_orders.groupBy("customer_id").agg(
    spark_max("order_date").alias("last_order_date"),
    count("order_id").alias("total_orders"),
    spark_sum("total_amount").alias("total_spent"),
    avg("total_amount").alias("avg_order_value")
)

rfm = rfm.withColumn(
    "recency_days", datediff(current_date(), col("last_order_date"))
)

display(rfm.limit(5))

each row is now one customer with their complete RFM base.
last_order_date is the most recent order date.
recency_days is how many days ago that was from today.
total_orders is how many times they ordered - frequency.
total_spent is their total spend across all orders - monetary value.
avg_order_value is their average spend per order.

these are the R, F and M of RFM - the three core features that 
Matuszelanski & Kopczewska (2022) identify as accounting for 
60-70% of churn model predictive power.
getting these right is the most important step in feature engineering.

**Clip and Round RFM Features**
- EDA says - clip total_orders at p99, round decimal columns.
- Paper says - monetary value needs to be clean and clipped to avoid outlier influence.

In [0]:
p99_orders = rfm.approxQuantile("total_orders", [0.99], 0.01)[0]
p99_spent  = rfm.approxQuantile("total_spent",  [0.99], 0.01)[0]

rfm = rfm.withColumn("total_orders", least(col("total_orders"), lit(p99_orders)))
rfm = rfm.withColumn("total_spent", least(col("total_spent"), lit(p99_spent)))
rfm = rfm.withColumn("total_spent", col("total_spent").cast("decimal(10,2)"))
rfm = rfm.withColumn("avg_order_value", col("avg_order_value").cast("decimal(10,2)"))

display(rfm.limit(5))

p99 means the 99th percentile.

total_orders and total_spent are now clipped at p99.
this means extreme outliers - bulk buyers, B2B orders, data anomalies - 
no longer dominate the feature space.

decimal values are rounded to 2 places - clean and readable.
this matters when we look at SHAP explanations later -
"total_spent = 1250.45" is much more readable than "1250.4523849..."

clipping at p99 is a standard preprocessing step.
the paper by Matuszelanski & Kopczewska (2022) explicitly handles 
outliers before training - they note that extreme values in payment 
amount distort the model's understanding of typical customer behavior.
a customer who spent Rs. 500,000 once is not a loyal customer -
they are an outlier that would confuse the model.

**Refund Features**
- EDA says - refund coverage is 60.6%, fill missing with 0. refund_rate = refund_count / total_orders.
- Paper says - perception features like dissatisfaction signals improve churn prediction on top of RFM.

In [0]:
refund_features = refunds_df.groupBy("customer_id").agg(
    count("refund_id").alias("refund_count")
)

rfm = rfm.join(refund_features, "customer_id", "left")
rfm = rfm.fillna(0, subset=["refund_count"])

rfm = rfm.withColumn(
    "refund_rate", (col("refund_count") / col("total_orders")).cast("decimal(5,2)")
)

display(rfm.select("customer_id", "total_orders", "refund_count", "refund_rate").limit(5))

refund_count is how many times a customer requested a refund.
refund_rate is refund_count divided by total_orders.

a customer with 3 orders and 2 refunds has refund_rate = 0.67 - 
they are refunding 67% of their orders. very unhappy customer.
a customer with 10 orders and 1 refund has refund_rate = 0.10 - 
occasional issue, not a pattern.

customers with no refunds get 0 - absence of refunds is itself 
a positive signal. they are satisfied with their orders.

this is a perception feature - it captures customer dissatisfaction 
through actual financial actions, not just words in a review.
a customer only requests a refund when they are genuinely unhappy -
it costs them time and effort.

the paper by Matuszelanski & Kopczewska (2022) groups this under 
perception features alongside review ratings. they found that 
perception features add meaningful signal on top of RFM -
especially when combined with behavioral features like recency.
a customer with high recency_days AND high refund_rate is almost 
certainly churned - the model will learn this combination.

**Cart Abandonment Feature**
- EDA says - cart abandonment rate fixed, clipped 0 to 1, use orders count not cart purchase action.
- Paper says - cart behavior is a core behavioral feature capturing customer purchase intent.

In [0]:
cart_adds = cart_df.filter(col("action") == "cart_add") \
    .groupBy("customer_id") \
    .agg(count("event_id").alias("cart_adds"))

cart_feat = cart_adds.join(
    rfm.select("customer_id", "total_orders"), "customer_id", "left"
)

cart_feat = cart_feat.withColumn(
    "abandon_rate",
    greatest(lit(0.0), least(lit(1.0),
        (col("cart_adds") - col("total_orders")) / col("cart_adds")
    ))
).select("customer_id", "abandon_rate")

rfm = rfm.join(cart_feat, "customer_id", "left")
rfm = rfm.fillna(0.0, subset=["abandon_rate"])

display(rfm.select("customer_id", "total_orders", "abandon_rate").limit(5))

abandon_rate is now correctly computed per customer.
it tells us what fraction of cart adds did not result in an order.

a customer with abandon_rate = 0.0 purchases everything they add to cart - 
highly committed buyer, low churn risk.
a customer with abandon_rate = 0.8 adds to cart frequently but 
rarely follows through - showing hesitation, higher churn risk.

we fixed two issues we found in EDA -
first we used order count from slv_orders instead of a purchase action 
from cart_events which did not exist.
second we clipped between 0 and 1 to remove negative values caused 
by direct purchases that bypassed the cart.

think of abandon_rate as a measure of purchase intent.
a customer whose abandon_rate is rising over time is losing 
their intent to buy - an early churn warning signal that shows up 
before recency_days crosses the 90 day threshold.

the paper by Matuszelanski & Kopczewska (2022) identifies cart behavior 
as a key behavioral feature that captures churn signals beyond basic RFM.
combined with recency_days it makes the model significantly more powerful 
at catching customers in the early stages of churning.

**Session Features**
- EDA says - session coverage is 99.9%, avg_pages_viewed and avg_session_duration are key engagement features. Clip session_duration at p99.
- Paper says - behavioral engagement features capture churn signals that transaction data alone misses.

In [0]:
session_feat = sessions_df.groupBy("customer_id").agg(
    avg("pages_viewed").alias("avg_pages_viewed"),
    avg("session_duration_seconds").alias("avg_session_duration"),
    count("session_id").alias("session_count")
)

rfm = rfm.join(session_feat, "customer_id", "left")
rfm = rfm.fillna(0, subset=["avg_pages_viewed", "avg_session_duration", "session_count"])

rfm = rfm.withColumn("avg_pages_viewed",    col("avg_pages_viewed").cast("decimal(5,2)"))
rfm = rfm.withColumn("avg_session_duration", col("avg_session_duration").cast("decimal(8,2)"))

display(rfm.select("customer_id", "avg_pages_viewed", "avg_session_duration", "session_count").limit(5))

avg_pages_viewed is the average number of pages a customer browses per session.
avg_session_duration is the average time they spend per session in seconds.
session_count is total number of sessions that customer has had.

these three features together paint a picture of customer engagement depth.
a highly engaged customer browses many pages and spends more time per session.
a disengaging customer opens the app briefly and leaves quickly.

as we discussed in EDA - declining session depth is a leading indicator 
of churn. it shows up weeks before the customer actually stops ordering.
recency_days catches churn late. session features catch it early.

customers with 0 in all three columns never had a recorded session -
they placed orders through other means like direct links or referrals.
we fill them with 0 - absence of session data is itself a signal.

the paper by Matuszelanski & Kopczewska (2022) groups session behavior 
under behavioral features alongside RFM. they found that behavioral 
features as a group account for 60-70% of model predictive power.
our session features add engagement depth that pure transaction data misses -
this is our upgrade on top of what the paper had available.

**Channel and Payment Features**
- EDA says - channel is 99% NULL in orders, take from sessions. payment_method from payments. device_type from sessions.
- Paper says - behavioral preference features like channel and payment method add meaningful signal.


In [0]:
# channel and device from sessions — orders.channel was 99% NULL
channel_feat = sessions_df.groupBy("customer_id").agg(
    avg(when(col("channel") == "organic_search", 1).otherwise(0)).alias("pct_organic"),
    avg(when(col("channel") == "social_media",   1).otherwise(0)).alias("pct_social")
)

# preferred payment from payments
payment_feat = payments_df.groupBy("customer_id", "payment_method") \
    .count().orderBy(col("count").desc()) \
    .dropDuplicates(["customer_id"]) \
    .select("customer_id", col("payment_method").alias("preferred_payment"))

rfm = rfm.join(channel_feat, "customer_id", "left")
rfm = rfm.join(payment_feat, "customer_id", "left")
rfm = rfm.fillna(0.0, subset=["pct_organic", "pct_social"])
rfm = rfm.fillna("unknown", subset=["preferred_payment"])

display(rfm.select("customer_id", "pct_organic", "pct_social", "preferred_payment").limit(5))

**Payment Investigation - Root Cause**

**Quick Check**

In [0]:
%sql
SELECT customer_id, payment_method, count(*) as cnt
FROM bharatmart.silver.slv_payments
WHERE customer_id IS NOT NULL
GROUP BY customer_id, payment_method
LIMIT 5

**Check Payment Method Distribution**

In [0]:
payments_df.filter(col("customer_id").isNotNull()) \
    .groupBy("payment_method") \
    .agg(count("payment_id").alias("cnt")) \
    .orderBy(col("cnt").desc()) \
    .show()

**Check Payment Coverage**

In [0]:
total_custs = rfm.select("customer_id").distinct().count()
pay_custs = payment_feat.select("customer_id").distinct().count()

print(f"customers in rfm : {total_custs:,}")
print(f"customers in payments : {pay_custs:,}")
print(f"missing payments : {total_custs - pay_custs:,}")

**Fix - Join Payments Through Orders**

In [0]:
# payments links to orders via order_id, orders has customer_id
order_payments = clean_orders.select("order_id", "customer_id") \
    .join(payments_df.select("order_id", "payment_method"), "order_id", "left")

payment_counts2 = order_payments.filter(col("payment_method").isNotNull()) \
    .groupBy("customer_id", "payment_method") \
    .agg(count("order_id").alias("cnt"))

window2 = Window.partitionBy("customer_id").orderBy(col("cnt").desc())

payment_feat2 = payment_counts2 \
    .withColumn("rn", row_number().over(window2)) \
    .filter(col("rn") == 1) \
    .select("customer_id", col("payment_method").alias("preferred_payment"))

print(f"customers with payment : {payment_feat2.count():,}")

**Channel and Payment Features - Clean Version**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# drop old broken columns first
rfm = rfm.drop("pct_organic", "pct_social", "preferred_payment")

# channel from sessions — orders.channel was 99% NULL
channel_feat = sessions_df.groupBy("customer_id").agg(
    avg(when(col("channel") == "organic_search", 1).otherwise(0)).alias("pct_organic"),
    avg(when(col("channel") == "social_media",   1).otherwise(0)).alias("pct_social")
)

# join payments through orders — direct customer_id in payments was only 13k
order_payments = clean_orders.select("order_id", "customer_id") \
    .join(payments_df.select("order_id", "payment_method"), "order_id", "left")

payment_counts = order_payments.filter(col("payment_method").isNotNull()) \
    .groupBy("customer_id", "payment_method") \
    .agg(count("order_id").alias("cnt"))

window = Window.partitionBy("customer_id").orderBy(col("cnt").desc())

payment_feat = payment_counts \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .select("customer_id", col("payment_method").alias("preferred_payment"))

rfm = rfm.join(channel_feat, "customer_id", "left").select(rfm['*'], channel_feat['pct_organic'].alias('pct_organic'), channel_feat['pct_social'].alias('pct_social'))
rfm = rfm.join(payment_feat, "customer_id", "left")
rfm = rfm.fillna(0.0,   subset=["pct_organic", "pct_social"])
rfm = rfm.fillna("UPI", subset=["preferred_payment"])

display(rfm.select("customer_id", "pct_organic", "pct_social", "preferred_payment").limit(5))

pct_organic is the share of sessions that came through organic search.
pct_social is the share of sessions that came through social media.
preferred_payment is the most frequently used payment method per customer.

channel was taken from sessions table - orders.channel was 99% NULL 
as confirmed in EDA and the silver layer SQL check.

preferred_payment was fixed by joining payments through orders via order_id.
direct join on customer_id in payments only had 13,666 records out of 92,107 -
that was causing 85% of customers to get unknown.
joining through order_id gives us correct payment method for all customers.

a customer who mostly comes via organic search is self-motivated -
they are actively seeking BharatMart without being pushed.
a customer who comes v

**Review Rating Feature**
- EDA says - avg_rating fill NULL with 3.0 (neutral). review coverage 92.6%.
- Paper says - raw rating weak alone but adds signal combined with refund_rate.

In [0]:
rating_feat = reviews_df.groupBy("customer_id").agg(
    avg("rating").alias("avg_rating")
)

rfm = rfm.join(rating_feat, "customer_id", "left")
rfm = rfm.fillna(3.0, subset=["avg_rating"])
rfm = rfm.withColumn("avg_rating", col("avg_rating").cast("decimal(3,1)"))

display(rfm.select("customer_id", "avg_rating").limit(5))

In [0]:
# Quick sanity check
reviews_df.filter((col("rating") < 1) | (col("rating") > 5)).count()

avg_rating is the average review rating a customer has given across all their orders.
values range from 1.0 to 5.0 - no anomalies confirmed.
customers with no reviews get 3.0 - neutral, no signal either way.

we fill with 3.0 not 0 because 0 would signal a very unhappy customer
which is wrong - they simply never reviewed, not that they were unhappy.

avg_rating alone is a weak churn predictor as the paper found -
Matuszelanski & Kopczewska (2022) showed that numeric review score 
had very low variable importance in the XGBoost model.
a customer who gives 1 star is almost as likely to reorder as 
someone who gives 5 stars - the rating alone does not explain churn.

however combined with refund_rate it adds signal -
a customer with low avg_rating AND high refund_rate is expressing 
dissatisfaction through two different channels simultaneously.
that combination is a much stronger churn signal than either alone.
XGBoost will discover this interaction automatically through its tree structure.

**Add Churn Label**
- EDA says - churn label needs total_orders >= 2, recency_days > 90.
- Paper says - churn label is the target variable, must be added before final feature table.

In [0]:
churn_labels = rfm.withColumn(
    "churn_label",
    when((col("recency_days") > 90) & (col("total_orders") >= 2), 1).otherwise(0)
)

churn_count  = churn_labels.filter(col("churn_label") == 1).count()
active_count = churn_labels.filter(col("churn_label") == 0).count()

print(f"churned : {churn_count:,}")
print(f"active  : {active_count:,}")

churn label matches exactly what we computed in EDA - 25,373 churned, 66,734 active.
this is our target variable - the y in the model.

churn_label = 1 - customer silent for 90+ days with at least 2 orders.
churn_label = 0 - customer is still active or is a first time buyer.

the fact that numbers match EDA exactly is a good sign -
our feature engineering pipeline is consistent with our analysis.
if the numbers had changed it would mean something went wrong in the joins.

this is exactly the behavioral silence definition of churn that 
Matuszelanski & Kopczewska (2022) use - no contract cancellation event,
just a customer who had a buying pattern and then stopped.
27.5% churn rate, scale_pos_weight = 2.6 for XGBoost confirmed.

**Encode Categorical Features**
- EDA says - encode categorical features: preferred_payment, need numeric for XGBoost.
- Paper says - categorical features need to be converted to numeric before model training.

In [0]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="preferred_payment",
    outputCol="payment_idx"
)

churn_labels = indexer.fit(churn_labels).transform(churn_labels)

display(churn_labels.select("preferred_payment", "payment_idx").distinct().orderBy("payment_idx"))

StringIndexer converts preferred_payment from text to numbers.
XGBoost cannot work with text values directly - it needs numbers.

UPI gets index 0 - it is the most frequent payment method.
StringIndexer assigns 0 to the most frequent, 1 to second most frequent and so on.
this ordering matches exactly what we saw in EDA - UPI dominated with 6,828 payments.

the encoding is:
UPI = 0, credit_card = 1, debit_card = 2, net_banking = 3, COD = 4, wallet = 5

this matters for SHAP explanations later -
when SHAP says payment_idx = 4 pushed a customer toward churn,
we know that means COD customers are at higher churn risk.

the paper by Matuszelanski & Kopczewska (2022) encodes all categorical 
features before training - they use one-hot encoding for product categories.
we use StringIndexer which is more efficient for XGBoost since it handles 
ordinal relationships naturally through its tree splits.

**Build Final Feature Table**
- EDA says - final feature table needs all 10 features + churn_label + customer_id.
- Paper says - final dataset should be clean, all numeric, ready for train/test split.

In [0]:
feature_cols = [
    "customer_id",
    "recency_days", "total_orders", "total_spent", "avg_order_value",
    "refund_count", "refund_rate",
    "abandon_rate",
    "avg_pages_viewed", "avg_session_duration", "session_count",
    "pct_organic", "pct_social",
    "avg_rating",
    "payment_idx",
    "churn_label"
]

final_df = churn_labels.select(feature_cols)

print(f"total customers : {final_df.count():,}")
print(f"total features  : {len(feature_cols) - 2}")
display(final_df.limit(5))

In [0]:
# Quick sanity checks
# check for any remaining nulls in final feature table
from pyspark.sql.functions import isnan

null_counts = final_df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in feature_cols if c != "customer_id"
]).toPandas()

print(null_counts.T)

92,107 customers with 14 clean features and zero nulls.
this is our complete feature table ready for model training.

the 14 features cover all three behavioral categories the paper identifies:

RFM core - recency_days, total_orders, total_spent, avg_order_value
these account for 60-70% of predictive power per Matuszelanski & Kopczewska (2022).

dissatisfaction signals - refund_count, refund_rate, avg_rating
capturing customer unhappiness through financial actions and ratings.

engagement depth - abandon_rate, avg_pages_viewed, avg_session_duration, session_count
capturing declining engagement before it shows up in recency.

behavioral preference - pct_organic, pct_social, payment_idx
capturing how and how the customer interacts with the platform.

zero nulls confirmed - every customer has a value

**Write Feature Table to ML Schema**

In [0]:
final_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bharatmart.ml.churn_features")

count = spark.table("bharatmart.ml.churn_features").count()
print(f"wrote {count:,} rows to bharatmart.ml.churn_features")

92,107 rows written to bharatmart.ml.churn_features in the ml schema.

we write to bharatmart.ml not bharatmart.gold because this is a 
model-specific feature table - not a business-facing gold table.
gold layer is for dashboards and business reporting.
ml schema is for model training artifacts and feature tables.

this table is now the single source of truth for Model 1 training.
every time we retrain the churn model we read from this table -
not from silver directly. this ensures consistency between training runs.

storing features in a dedicated table also means other models can reuse them.
the campaign targeting model can read the same churn_features table 
instead of recomputing the same features from scratch.

the paper by Matuszelanski & Kopczewska (2022) emphasizes reproducibility -
a churn model is only useful in production if it can be retrained 
consistently on the same feature definitions.
writing to a versioned Delta table gives us exactly that - 
Delta's time travel means we can always go back to any previous version.

**feature engineering is complete. here is everything we built and why.**

FEATURE COUNT - 14 features for 92,107 customers, zero nulls.

originally EDA checklist said 10 features.
channel became pct_organic and pct_social - two proportion features 
instead of one categorical. this is actually smarter because it preserves 
the mix of channels per customer, not just the dominant one.
a customer who is 70% organic and 30% social is different from one who 
is 100% social - StringIndexer would have lost that nuance.

THE 14 FEATURES:

RFM core - recency_days, total_orders, total_spent, avg_order_value
paper says these account for 60-70% of predictive power.

dissatisfaction signals - refund_count, refund_rate, avg_rating
refund_rate is stronger than avg_rating - money talks louder than stars.

engagement depth - abandon_rate, avg_pages_viewed, avg_session_duration, session_count
these catch churn early - before recency_days crosses 90 days.

behavioral preference - pct_organic, pct_social, payment_idx
how and where the customer interacts with BharatMart.

PAYMENT INDEX MAPPING - save this for SHAP explanations:
UPI = 0, credit_card = 1, debit_card = 2
net_banking = 3, COD = 4, wallet = 5

when SHAP says payment_idx = 4.0 pushed a customer toward churn -
that means COD customers churn more. say it confidently in the demo.

KEY FIXES MADE DURING ENGINEERING:
- total_amount recomputed - was 99% NULL in silver
- channel taken from sessions - was 99% NULL in orders
- abandon_rate fixed twice - wrong source and negative values
- payment join fixed - went through order_id not customer_id directly
  (direct join only had 13,666 of 92,107 customers)

lesson learned - always validate output makes business sense.
unknown, 0 where it should not be, or low counts are red flags.
never assume a fill value is correct without finding the root cause.

written to bharatmart.ml.churn_features - single source of truth for model training.
based on Matuszelanski & Kopczewska (2022) - behavioral features are the 
foundation of churn prediction, our 14 features cover all categories 
the paper identifies and go beyond what their Olist dataset had available.